# Notebook 02 — TF-IDF + Logistic Regression Baseline

**Purpose**: Train and evaluate TF-IDF + Logistic Regression pipelines for:
- Task A: Binary classification (sarcastic vs non-sarcastic)
- Task B: Sarcasm type classification (6-class)

**Prerequisite**: Run `01_data_preparation.ipynb` first.

**Outputs**:
- `outputs/classical/tfidf_lr/best_config_binary.json`
- `outputs/classical/tfidf_lr/best_config_type.json`
- `outputs/classical/tfidf_lr/metrics_binary.json`
- `outputs/classical/tfidf_lr/metrics_type.json`
- Predictions CSV + confusion matrices PNG

In [ ]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Robust project-root finder ─────────────────────────────────────────────
def _find_project_root() -> Path:
    """Walk up from cwd until we find outputs/splits or another project marker."""
    markers = [
        Path("outputs") / "splits",
        Path("outputs") / "datasets",
        Path("data") / "processed",
        Path("notebooks"),
    ]
    for root in [Path.cwd()] + list(Path.cwd().parents):
        for marker in markers:
            if (root / marker).exists():
                return root
    return Path.cwd().parent  # fallback

ROOT    = _find_project_root()
SPLITS  = ROOT / "outputs" / "splits"
OUT_DIR = ROOT / "outputs" / "classical" / "tfidf_lr"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root    : {ROOT}")
print(f"Output directory: {OUT_DIR}")


## Helper Functions

In [ ]:
def load_splits(task: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load train/val/test CSVs for a given task ('binary' or 'type')."""
    train = pd.read_csv(SPLITS / f"train_{task}.csv")
    val   = pd.read_csv(SPLITS / f"val_{task}.csv")
    test  = pd.read_csv(SPLITS / f"test_{task}.csv")
    return train, val, test


def evaluate(
    model,
    X: list[str],
    y_true: list,
    label_names: list[str] | None = None,
    split_name: str = "",
) -> dict:
    """Compute classification metrics and return as dict."""
    y_pred = model.predict(X)
    metrics = {
        "split"           : split_name,
        "accuracy"        : accuracy_score(y_true, y_pred),
        "precision_macro" : precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro"    : recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro"        : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted"     : f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "report"          : classification_report(y_true, y_pred, target_names=label_names,
                                                  zero_division=0, output_dict=True),
    }
    print(f"\n[{split_name}] Accuracy={metrics['accuracy']:.4f}  "
          f"Macro-F1={metrics['f1_macro']:.4f}  Weighted-F1={metrics['f1_weighted']:.4f}")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    return metrics, y_pred


def save_confusion_matrix(
    y_true, y_pred, label_names: list[str], out_path: Path, title: str = ""
) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))
    fig, ax = plt.subplots(figsize=(max(6, len(label_names)), max(5, len(label_names))))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_names, yticklabels=label_names, ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path.relative_to(ROOT)}")


def save_predictions(
    df: pd.DataFrame, y_pred, label_col: str, out_path: Path
) -> None:
    out = df[["sample_id", "pair_id", "group_id", "text", label_col]].copy()
    out["predicted"] = y_pred
    out["correct"]   = (out[label_col] == out["predicted"]).astype(int)
    out.to_csv(out_path, index=False)
    print(f"Saved: {out_path.relative_to(ROOT)}")

## Task A â€” Binary Classification

In [ ]:
# â”€â”€ Load binary splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_bin, val_bin, test_bin = load_splits("binary")

# Combine train+val for cross-validation, keep test for final eval
trainval_bin = pd.concat([train_bin, val_bin], ignore_index=True)

X_trainval = trainval_bin["text"].tolist()
y_trainval = trainval_bin["binary_label"].tolist()
groups_tv  = trainval_bin["group_id"].tolist()

X_train = train_bin["text"].tolist()
y_train = train_bin["binary_label"].tolist()

X_val   = val_bin["text"].tolist()
y_val   = val_bin["binary_label"].tolist()

X_test  = test_bin["text"].tolist()
y_test  = test_bin["binary_label"].tolist()

label_names_bin = ["non-sarcastic", "sarcastic"]

print(f"Train+Val: {len(X_trainval):,}  |  Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

In [ ]:
# â”€â”€ Define pipeline and grid â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pipe_bin = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, lowercase=True)),
    ("lr",    LogisticRegression(max_iter=1000, random_state=SEED, solver="lbfgs")),
])

param_grid_bin = {
    "tfidf__ngram_range" : [(1, 1), (1, 2)],
    "tfidf__min_df"      : [2, 3, 5],
    "tfidf__max_features": [None, 50_000],
    "lr__C"              : [0.1, 1.0, 3.0],
}

cv_bin = GroupKFold(n_splits=5)

gs_bin = GridSearchCV(
    pipe_bin, param_grid_bin,
    cv=cv_bin, scoring="f1_macro",
    n_jobs=-1, verbose=1, refit=True,
)

print(f"Grid size: {len(gs_bin.param_grid)} param combos â†’ {5 * 2*3*2*3} fits")
print("Running grid search...")
gs_bin.fit(X_trainval, y_trainval, groups=groups_tv)

print(f"\nBest params  : {gs_bin.best_params_}")
print(f"Best CV F1   : {gs_bin.best_score_:.4f}")

In [ ]:
# â”€â”€ Save best config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_cfg_bin = {"task": "binary", "model": "TF-IDF + LR", "seed": SEED,
                "best_params": gs_bin.best_params_, "cv_f1_macro": gs_bin.best_score_}
with open(OUT_DIR / "best_config_binary.json", "w") as f:
    json.dump(best_cfg_bin, f, indent=2)
print("Saved: outputs/classical/tfidf_lr/best_config_binary.json")

In [ ]:
# â”€â”€ Evaluate on val and test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_bin = gs_bin.best_estimator_

val_metrics_bin,  y_val_pred_bin  = evaluate(best_bin, X_val,  y_val,  label_names_bin, "Val")
test_metrics_bin, y_test_pred_bin = evaluate(best_bin, X_test, y_test, label_names_bin, "Test")

all_metrics_bin = {"val": val_metrics_bin, "test": test_metrics_bin}

# Remove classification_report dict (too nested for JSON)
for split_m in all_metrics_bin.values():
    split_m.pop("report", None)

with open(OUT_DIR / "metrics_binary.json", "w") as f:
    json.dump(all_metrics_bin, f, indent=2)
print("Saved: outputs/classical/tfidf_lr/metrics_binary.json")

In [ ]:
# â”€â”€ Confusion matrices â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
save_confusion_matrix(
    y_val, y_val_pred_bin, label_names_bin,
    OUT_DIR / "confusion_matrix_binary_val.png",
    "TF-IDF + LR â€” Binary (Val)"
)
save_confusion_matrix(
    y_test, y_test_pred_bin, label_names_bin,
    OUT_DIR / "confusion_matrix_binary_test.png",
    "TF-IDF + LR â€” Binary (Test)"
)

In [ ]:
# â”€â”€ Save predictions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
save_predictions(val_bin,  y_val_pred_bin,  "binary_label", OUT_DIR / "predictions_val_binary.csv")
save_predictions(test_bin, y_test_pred_bin, "binary_label", OUT_DIR / "predictions_test_binary.csv")

In [ ]:
# â”€â”€ Feature importance: top discriminative terms â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tfidf_vocab = best_bin.named_steps["tfidf"].get_feature_names_out()
lr_coefs    = best_bin.named_steps["lr"].coef_[0]  # binary: shape (n_features,)

n_top = 20
top_pos_idx = np.argsort(lr_coefs)[-n_top:][::-1]
top_neg_idx = np.argsort(lr_coefs)[:n_top]

top_positive = [(tfidf_vocab[i], lr_coefs[i]) for i in top_pos_idx]
top_negative = [(tfidf_vocab[i], lr_coefs[i]) for i in top_neg_idx]

print("Top 20 sarcastic indicators (positive coef):")
for term, coef in top_positive:
    print(f"  {term:35s}  {coef:+.4f}")

print("\nTop 20 non-sarcastic indicators (negative coef):")
for term, coef in top_negative:
    print(f"  {term:35s}  {coef:+.4f}")

In [ ]:
# â”€â”€ Feature importance plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

terms_pos, coefs_pos = zip(*top_positive)
axes[0].barh(list(reversed(terms_pos)), list(reversed(coefs_pos)), color="steelblue")
axes[0].set_title("Top 20 â€” Sarcastic (positive)", fontsize=13)
axes[0].set_xlabel("LR Coefficient")

terms_neg, coefs_neg = zip(*top_negative)
axes[1].barh(list(reversed(terms_neg)), list(reversed(coefs_neg)), color="coral")
axes[1].set_title("Top 20 â€” Non-Sarcastic (negative)", fontsize=13)
axes[1].set_xlabel("LR Coefficient")

plt.suptitle("TF-IDF + LR: Feature Importance (Binary Task)", fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance_binary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/classical/tfidf_lr/feature_importance_binary.png")

## Task B â€” Sarcasm Type Classification

In [ ]:
# â”€â”€ Load type splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_type, val_type, test_type = load_splits("type")

# Encode string labels to integers
STRATEGY_LABELS = sorted(train_type["type_label"].unique())
label2id = {lab: i for i, lab in enumerate(STRATEGY_LABELS)}
id2label = {i: lab for lab, i in label2id.items()}

def encode_labels(df: pd.DataFrame) -> list[int]:
    return [label2id[l] for l in df["type_label"]]

trainval_type = pd.concat([train_type, val_type], ignore_index=True)

X_trainval_t = trainval_type["text"].tolist()
y_trainval_t = encode_labels(trainval_type)
groups_tv_t  = trainval_type["group_id"].tolist()

X_train_t = train_type["text"].tolist()
y_train_t = encode_labels(train_type)
X_val_t   = val_type["text"].tolist()
y_val_t   = encode_labels(val_type)
X_test_t  = test_type["text"].tolist()
y_test_t  = encode_labels(test_type)

print(f"Strategies: {STRATEGY_LABELS}")
print(f"Train+Val: {len(X_trainval_t):,}  Train: {len(X_train_t):,}  Val: {len(X_val_t):,}  Test: {len(X_test_t):,}")

In [ ]:
# â”€â”€ Define pipeline and grid (with class_weight) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pipe_type = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, lowercase=True)),
    ("lr",    LogisticRegression(max_iter=1000, random_state=SEED, solver="lbfgs",
                                 multi_class="auto")),
])

param_grid_type = {
    "tfidf__ngram_range" : [(1, 1), (1, 2)],
    "tfidf__min_df"      : [2, 3, 5],
    "tfidf__max_features": [None, 50_000],
    "lr__C"              : [0.1, 1.0, 3.0],
    "lr__class_weight"   : [None, "balanced"],
}

cv_type = GroupKFold(n_splits=5)

gs_type = GridSearchCV(
    pipe_type, param_grid_type,
    cv=cv_type, scoring="f1_macro",
    n_jobs=-1, verbose=1, refit=True,
)

print("Running grid search for type task...")
gs_type.fit(X_trainval_t, y_trainval_t, groups=groups_tv_t)

print(f"\nBest params  : {gs_type.best_params_}")
print(f"Best CV F1   : {gs_type.best_score_:.4f}")

In [ ]:
# â”€â”€ Save best config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_cfg_type = {"task": "type", "model": "TF-IDF + LR", "seed": SEED,
                 "label_names": STRATEGY_LABELS, "label2id": label2id,
                 "best_params": gs_type.best_params_, "cv_f1_macro": gs_type.best_score_}
with open(OUT_DIR / "best_config_type.json", "w") as f:
    json.dump(best_cfg_type, f, indent=2)
print("Saved: outputs/classical/tfidf_lr/best_config_type.json")

In [ ]:
# â”€â”€ Evaluate on val and test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_type = gs_type.best_estimator_

val_metrics_type,  y_val_pred_type  = evaluate(best_type, X_val_t,  y_val_t,  STRATEGY_LABELS, "Val")
test_metrics_type, y_test_pred_type = evaluate(best_type, X_test_t, y_test_t, STRATEGY_LABELS, "Test")

all_metrics_type = {"val": val_metrics_type, "test": test_metrics_type}
for split_m in all_metrics_type.values():
    split_m.pop("report", None)

with open(OUT_DIR / "metrics_type.json", "w") as f:
    json.dump(all_metrics_type, f, indent=2)
print("Saved: outputs/classical/tfidf_lr/metrics_type.json")

In [ ]:
# â”€â”€ Confusion matrices â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
save_confusion_matrix(
    y_val_t, y_val_pred_type, STRATEGY_LABELS,
    OUT_DIR / "confusion_matrix_type_val.png",
    "TF-IDF + LR â€” Type (Val)"
)
save_confusion_matrix(
    y_test_t, y_test_pred_type, STRATEGY_LABELS,
    OUT_DIR / "confusion_matrix_type_test.png",
    "TF-IDF + LR â€” Type (Test)"
)

In [ ]:
# â”€â”€ Save predictions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Decode integer predictions back to string labels
val_type_copy  = val_type.copy();  val_type_copy["type_label"]  = y_val_t
test_type_copy = test_type.copy(); test_type_copy["type_label"] = y_test_t

val_type_copy["predicted_id"]    = y_val_pred_type
val_type_copy["predicted_label"] = [id2label[i] for i in y_val_pred_type]
val_type_copy["true_label"]      = [id2label[i] for i in y_val_t]
val_type_copy["correct"]         = (val_type_copy["type_label"] == val_type_copy["predicted_id"]).astype(int)

test_type_copy["predicted_id"]    = y_test_pred_type
test_type_copy["predicted_label"] = [id2label[i] for i in y_test_pred_type]
test_type_copy["true_label"]      = [id2label[i] for i in y_test_t]
test_type_copy["correct"]         = (test_type_copy["type_label"] == test_type_copy["predicted_id"]).astype(int)

val_type_copy.to_csv( OUT_DIR / "predictions_val_type.csv",  index=False)
test_type_copy.to_csv(OUT_DIR / "predictions_test_type.csv", index=False)
print("Saved predictions_val_type.csv and predictions_test_type.csv")

## Char N-gram Variant (Optional)

In [ ]:
# Char n-gram model for binary task (stylistic cues)
pipe_char = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                              sublinear_tf=True, lowercase=True, max_features=50_000)),
    ("lr",    LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)),
])

param_grid_char = {"lr__C": [0.1, 1.0, 3.0]}
gs_char = GridSearchCV(pipe_char, param_grid_char, cv=GroupKFold(5),
                       scoring="f1_macro", n_jobs=-1)
gs_char.fit(X_trainval, y_trainval, groups=groups_tv)

print(f"Char n-gram best C      : {gs_char.best_params_}")
print(f"Char n-gram best CV F1  : {gs_char.best_score_:.4f}")
print(f"Word n-gram best CV F1  : {gs_bin.best_score_:.4f}")
print()
char_metrics, _ = evaluate(gs_char.best_estimator_, X_test, y_test, label_names_bin, "Test (char)")

## Summary

In [ ]:
print("====== TF-IDF + LR RESULTS SUMMARY ======")
print()
print("Task A â€” Binary Classification (Test Set):")
print(f"  Accuracy        : {test_metrics_bin['accuracy']:.4f}")
print(f"  Macro-F1        : {test_metrics_bin['f1_macro']:.4f}")
print(f"  Weighted-F1     : {test_metrics_bin['f1_weighted']:.4f}")
print()
print("Task B â€” Type Classification (Test Set):")
print(f"  Accuracy        : {test_metrics_type['accuracy']:.4f}")
print(f"  Macro-F1        : {test_metrics_type['f1_macro']:.4f}")
print(f"  Weighted-F1     : {test_metrics_type['f1_weighted']:.4f}")
print()
print("Best hyperparameters:")
print("  Binary:", gs_bin.best_params_)
print("  Type:  ", gs_type.best_params_)